# Quality Stream: All HCPCS per NPI across Medicare & Medicaid

**Right Problem**: Join Medicare + Medicaid on NPI to get a unified view of all procedure codes (HCPCS/CPT), volumes, and spending per provider across both payers.

**Datasets**:
- `MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv` — Medicare Physician & Other Practitioners (Provider-Service level)
- `medicaid-provider-spending.parquet` — Medicaid Provider Spending (claim-level aggregates)

**Technique**: Streaming in 50K-row chunks for ~3GB files

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import plotly.express as px
import plotly.graph_objects as go
from collections import defaultdict
import os

CHUNK_SIZE = 50_000
DATA_DIR = "datasets"
MEDICARE_FILE = os.path.join(DATA_DIR, "MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv")
MEDICAID_FILE = os.path.join(DATA_DIR, "medicaid-provider-spending.parquet")

print("Libraries loaded. Chunk size:", f"{CHUNK_SIZE:,}")

---
## 1. Exploratory Data Analysis (EDA)

### 1.1 Medicare — Schema, Shape & Sample

In [ ]:
# Medicare: Get schema, shape, and sample via chunked read
medicare_total_rows = 0
medicare_sample = None

for chunk in pd.read_csv(MEDICARE_FILE, chunksize=CHUNK_SIZE, low_memory=False):
    medicare_total_rows += len(chunk)
    if medicare_sample is None:
        medicare_sample = chunk.head(5)
        medicare_dtypes = chunk.dtypes
        medicare_columns = list(chunk.columns)
        medicare_ncols = len(chunk.columns)

print(f"Medicare shape: ({medicare_total_rows:,} rows, {medicare_ncols} columns)")
print(f"\nColumns ({medicare_ncols}):")
for i, col in enumerate(medicare_columns):
    print(f"  {i+1}. {col} — {medicare_dtypes[col]}")
print(f"\nSample (first 5 rows):")
medicare_sample

### 1.2 Medicaid — Schema, Shape & Sample

In [ ]:
# Medicaid: Get schema and shape from parquet metadata (no full scan needed)
pf = pq.ParquetFile(MEDICAID_FILE)
medicaid_total_rows = pf.metadata.num_rows
medicaid_schema = pf.schema_arrow
medicaid_columns = [field.name for field in medicaid_schema]
medicaid_ncols = len(medicaid_columns)

# Read a small batch for sample
medicaid_sample = next(pf.iter_batches(batch_size=5))
medicaid_sample_df = pd.DataFrame(medicaid_sample.to_pydict())

print(f"Medicaid shape: ({medicaid_total_rows:,} rows, {medicaid_ncols} columns)")
print(f"\nSchema:")
for i, field in enumerate(medicaid_schema):
    print(f"  {i+1}. {field.name} — {field.type}")
print(f"\nSample (first 5 rows):")
medicaid_sample_df

### 1.3 Null Values Analysis

In [ ]:
# Medicare: Compute null proportions via chunked read
medicare_null_counts = defaultdict(int)
medicare_row_count = 0

for chunk in pd.read_csv(MEDICARE_FILE, chunksize=CHUNK_SIZE, low_memory=False):
    medicare_row_count += len(chunk)
    for col in chunk.columns:
        medicare_null_counts[col] += chunk[col].isna().sum()

medicare_nulls = pd.DataFrame({
    "column": list(medicare_null_counts.keys()),
    "null_count": list(medicare_null_counts.values()),
    "null_pct": [v / medicare_row_count * 100 for v in medicare_null_counts.values()]
}).sort_values("null_pct", ascending=False).reset_index(drop=True)

print(f"Medicare — Null proportions ({medicare_row_count:,} rows):")
medicare_nulls

In [ ]:
# Medicaid: Compute null proportions via chunked parquet read
medicaid_null_counts = defaultdict(int)
medicaid_row_count = 0

for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
    chunk = pd.DataFrame(batch.to_pydict())
    medicaid_row_count += len(chunk)
    for col in chunk.columns:
        medicaid_null_counts[col] += chunk[col].isna().sum()

medicaid_nulls = pd.DataFrame({
    "column": list(medicaid_null_counts.keys()),
    "null_count": list(medicaid_null_counts.values()),
    "null_pct": [v / medicaid_row_count * 100 for v in medicaid_null_counts.values()]
}).sort_values("null_pct", ascending=False).reset_index(drop=True)

print(f"Medicaid — Null proportions ({medicaid_row_count:,} rows):")
medicaid_nulls

### 1.4 Data Types Summary

In [ ]:
# Side-by-side data type summary
print("=" * 60)
print("MEDICARE Data Types")
print("=" * 60)
for col in medicare_columns:
    print(f"  {col:45s} {str(medicare_dtypes[col]):>12s}")

print(f"\n{'=' * 60}")
print("MEDICAID Data Types")
print("=" * 60)
for field in medicaid_schema:
    print(f"  {field.name:45s} {str(field.type):>12s}")

### 1.5 Charts — Getting a Sense of the Data

In [ ]:
# Chart 1: Top 20 Provider Types by count (Medicare) — chunked read
provider_type_counts = defaultdict(int)

for chunk in pd.read_csv(MEDICARE_FILE, chunksize=CHUNK_SIZE, low_memory=False, usecols=["Rndrng_Prvdr_Type"]):
    counts = chunk["Rndrng_Prvdr_Type"].value_counts()
    for ptype, count in counts.items():
        provider_type_counts[ptype] += count

pt_df = pd.DataFrame(
    sorted(provider_type_counts.items(), key=lambda x: x[1], reverse=True)[:20],
    columns=["Provider Type", "Record Count"]
)

fig = px.bar(pt_df, x="Record Count", y="Provider Type", orientation="h",
             title="Medicare: Top 20 Provider Types by Record Count",
             color="Record Count", color_continuous_scale="Blues")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [ ]:
# Chart 2: Top 20 HCPCS codes by total services (Medicare)
hcpcs_svc_counts = defaultdict(float)

for chunk in pd.read_csv(MEDICARE_FILE, chunksize=CHUNK_SIZE, low_memory=False, usecols=["HCPCS_Cd", "Tot_Srvcs"]):
    grouped = chunk.groupby("HCPCS_Cd")["Tot_Srvcs"].sum()
    for code, total in grouped.items():
        hcpcs_svc_counts[code] += total

hcpcs_df = pd.DataFrame(
    sorted(hcpcs_svc_counts.items(), key=lambda x: x[1], reverse=True)[:20],
    columns=["HCPCS Code", "Total Services"]
)

fig = px.bar(hcpcs_df, x="Total Services", y="HCPCS Code", orientation="h",
             title="Medicare: Top 20 HCPCS Codes by Total Services",
             color="Total Services", color_continuous_scale="Greens")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [ ]:
# Chart 3: Top 20 HCPCS codes by total claims (Medicaid) — chunked parquet
medicaid_hcpcs_claims = defaultdict(int)

for batch in pf.iter_batches(batch_size=CHUNK_SIZE, columns=["HCPCS_CODE", "TOTAL_CLAIMS"]):
    chunk = pd.DataFrame(batch.to_pydict())
    grouped = chunk.groupby("HCPCS_CODE")["TOTAL_CLAIMS"].sum()
    for code, total in grouped.items():
        medicaid_hcpcs_claims[code] += total

medicaid_hcpcs_df = pd.DataFrame(
    sorted(medicaid_hcpcs_claims.items(), key=lambda x: x[1], reverse=True)[:20],
    columns=["HCPCS Code", "Total Claims"]
)

fig = px.bar(medicaid_hcpcs_df, x="Total Claims", y="HCPCS Code", orientation="h",
             title="Medicaid: Top 20 HCPCS Codes by Total Claims",
             color="Total Claims", color_continuous_scale="Oranges")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [ ]:
# Chart 4: Medicare Average Payment Distribution (sampled)
payment_sample = []
rows_sampled = 0

for chunk in pd.read_csv(MEDICARE_FILE, chunksize=CHUNK_SIZE, low_memory=False, usecols=["Avg_Mdcr_Pymt_Amt"]):
    payment_sample.append(chunk.sample(min(1000, len(chunk)), random_state=42))
    rows_sampled += len(chunk)
    if rows_sampled > 2_000_000:
        break

payment_df = pd.concat(payment_sample)

fig = px.histogram(payment_df, x="Avg_Mdcr_Pymt_Amt", nbins=100,
                   title="Medicare: Distribution of Avg Payment Amount (sampled)",
                   labels={"Avg_Mdcr_Pymt_Amt": "Avg Medicare Payment ($)"},
                   color_discrete_sequence=["#636EFA"])
fig.update_xaxes(range=[0, 500])
fig.show()

In [ ]:
# Chart 5: Medicaid Total Paid Distribution (sampled)
medicaid_paid_sample = []
rows_sampled = 0

for batch in pf.iter_batches(batch_size=CHUNK_SIZE, columns=["TOTAL_PAID"]):
    chunk = pd.DataFrame(batch.to_pydict())
    medicaid_paid_sample.append(chunk.sample(min(1000, len(chunk)), random_state=42))
    rows_sampled += len(chunk)
    if rows_sampled > 2_000_000:
        break

medicaid_paid_df = pd.concat(medicaid_paid_sample)

fig = px.histogram(medicaid_paid_df, x="TOTAL_PAID", nbins=100,
                   title="Medicaid: Distribution of Total Paid (sampled)",
                   labels={"TOTAL_PAID": "Total Paid ($)"},
                   color_discrete_sequence=["#EF553B"])
fig.update_xaxes(range=[0, 50000])
fig.show()

---
## 2. Data Preprocessing & Cleaning

Harmonize NPI and HCPCS fields to string, aggregate metrics per NPI-HCPCS pair for each payer.

In [ ]:
# Aggregate Medicare by NPI + HCPCS (chunked)
# Keep: NPI, HCPCS, provider info (from first occurrence), summed metrics

medicare_agg = defaultdict(lambda: {
    "tot_benes": 0, "tot_srvcs": 0, "tot_bene_day_srvcs": 0,
    "sum_pymt": 0.0, "sum_chrg": 0.0, "sum_alowd": 0.0, "count": 0
})
medicare_provider_info = {}  # NPI -> provider info (captured once)
medicare_hcpcs_desc = {}  # HCPCS -> description (captured once)

use_cols = [
    "Rndrng_NPI", "HCPCS_Cd", "HCPCS_Desc",
    "Rndrng_Prvdr_Last_Org_Name", "Rndrng_Prvdr_First_Name",
    "Rndrng_Prvdr_Type", "Rndrng_Prvdr_State_Abrvtn", "Rndrng_Prvdr_City",
    "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
    "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt"
]

chunks_processed = 0
for chunk in pd.read_csv(MEDICARE_FILE, chunksize=CHUNK_SIZE, low_memory=False, usecols=use_cols):
    # Harmonize NPI to string, strip whitespace
    chunk["Rndrng_NPI"] = chunk["Rndrng_NPI"].astype(str).str.strip()
    chunk["HCPCS_Cd"] = chunk["HCPCS_Cd"].astype(str).str.strip().str.upper()

    # Ensure numeric columns
    for num_col in ["Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs", "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt"]:
        chunk[num_col] = pd.to_numeric(chunk[num_col], errors="coerce").fillna(0)

    for _, row in chunk.iterrows():
        npi = row["Rndrng_NPI"]
        hcpcs = row["HCPCS_Cd"]
        key = (npi, hcpcs)

        rec = medicare_agg[key]
        rec["tot_benes"] += row["Tot_Benes"]
        rec["tot_srvcs"] += row["Tot_Srvcs"]
        rec["tot_bene_day_srvcs"] += row["Tot_Bene_Day_Srvcs"]
        rec["sum_pymt"] += row["Avg_Mdcr_Pymt_Amt"] * row["Tot_Srvcs"]
        rec["sum_chrg"] += row["Avg_Sbmtd_Chrg"] * row["Tot_Srvcs"]
        rec["sum_alowd"] += row["Avg_Mdcr_Alowd_Amt"] * row["Tot_Srvcs"]
        rec["count"] += 1

        # Capture provider info once per NPI
        if npi not in medicare_provider_info:
            medicare_provider_info[npi] = {
                "provider_name": f"{row['Rndrng_Prvdr_First_Name']} {row['Rndrng_Prvdr_Last_Org_Name']}".strip(),
                "provider_type": row["Rndrng_Prvdr_Type"],
                "provider_state": row["Rndrng_Prvdr_State_Abrvtn"],
                "provider_city": row["Rndrng_Prvdr_City"]
            }

        # Capture HCPCS description once
        if hcpcs not in medicare_hcpcs_desc and pd.notna(row["HCPCS_Desc"]):
            medicare_hcpcs_desc[hcpcs] = row["HCPCS_Desc"]

    chunks_processed += 1
    if chunks_processed % 20 == 0:
        print(f"  Medicare: processed {chunks_processed * CHUNK_SIZE:,} rows...")

# Build Medicare DataFrame
medicare_records = []
for (npi, hcpcs), vals in medicare_agg.items():
    medicare_records.append({
        "npi": npi,
        "hcpcs_code": hcpcs,
        "medicare_total_benes": int(vals["tot_benes"]),
        "medicare_total_srvcs": int(vals["tot_srvcs"]),
        "medicare_total_pymt": round(vals["sum_pymt"], 2),
        "medicare_total_chrg": round(vals["sum_chrg"], 2),
        "medicare_total_alowd": round(vals["sum_alowd"], 2)
    })

df_medicare = pd.DataFrame(medicare_records)
print(f"\nMedicare aggregated: {len(df_medicare):,} NPI-HCPCS pairs from {len(medicare_provider_info):,} unique NPIs")
df_medicare.head()

In [ ]:
# Aggregate Medicaid by NPI + HCPCS (chunked parquet)
# Using SERVICING_PROVIDER_NPI_NUM as the primary NPI for the join

medicaid_agg = defaultdict(lambda: {
    "total_benes": 0, "total_claims": 0, "total_paid": 0.0
})

chunks_processed = 0
for batch in pf.iter_batches(batch_size=CHUNK_SIZE, columns=["SERVICING_PROVIDER_NPI_NUM", "HCPCS_CODE", "TOTAL_UNIQUE_BENEFICIARIES", "TOTAL_CLAIMS", "TOTAL_PAID"]):
    chunk = pd.DataFrame(batch.to_pydict())

    # Harmonize NPI and HCPCS to string, strip whitespace
    chunk["SERVICING_PROVIDER_NPI_NUM"] = chunk["SERVICING_PROVIDER_NPI_NUM"].astype(str).str.strip()
    chunk["HCPCS_CODE"] = chunk["HCPCS_CODE"].astype(str).str.strip().str.upper()

    # Ensure numeric columns
    for num_col in ["TOTAL_UNIQUE_BENEFICIARIES", "TOTAL_CLAIMS", "TOTAL_PAID"]:
        chunk[num_col] = pd.to_numeric(chunk[num_col], errors="coerce").fillna(0)

    grouped = chunk.groupby(["SERVICING_PROVIDER_NPI_NUM", "HCPCS_CODE"]).agg({
        "TOTAL_UNIQUE_BENEFICIARIES": "sum",
        "TOTAL_CLAIMS": "sum",
        "TOTAL_PAID": "sum"
    })

    for (npi, hcpcs), row in grouped.iterrows():
        key = (npi, hcpcs)
        rec = medicaid_agg[key]
        rec["total_benes"] += int(row["TOTAL_UNIQUE_BENEFICIARIES"])
        rec["total_claims"] += int(row["TOTAL_CLAIMS"])
        rec["total_paid"] += float(row["TOTAL_PAID"])

    chunks_processed += 1
    if chunks_processed % 100 == 0:
        print(f"  Medicaid: processed {chunks_processed * CHUNK_SIZE:,} rows...")

# Build Medicaid DataFrame
medicaid_records = []
for (npi, hcpcs), vals in medicaid_agg.items():
    medicaid_records.append({
        "npi": npi,
        "hcpcs_code": hcpcs,
        "medicaid_total_benes": vals["total_benes"],
        "medicaid_total_claims": vals["total_claims"],
        "medicaid_total_paid": round(vals["total_paid"], 2)
    })

df_medicaid = pd.DataFrame(medicaid_records)
print(f"\nMedicaid aggregated: {len(df_medicaid):,} NPI-HCPCS pairs from {df_medicaid['npi'].nunique():,} unique NPIs")
df_medicaid.head()

---
## 3. Join: All HCPCS per NPI across Both Payers

In [ ]:
# Full outer join on NPI + HCPCS code to capture all procedures from both payers
df_joined = pd.merge(
    df_medicare,
    df_medicaid,
    on=["npi", "hcpcs_code"],
    how="outer",
    indicator=True
)

# Add provider info from Medicare lookup
df_joined["provider_name"] = df_joined["npi"].map(lambda x: medicare_provider_info.get(x, {}).get("provider_name", ""))
df_joined["provider_type"] = df_joined["npi"].map(lambda x: medicare_provider_info.get(x, {}).get("provider_type", ""))
df_joined["provider_state"] = df_joined["npi"].map(lambda x: medicare_provider_info.get(x, {}).get("provider_state", ""))
df_joined["hcpcs_desc"] = df_joined["hcpcs_code"].map(lambda x: medicare_hcpcs_desc.get(x, ""))

# Label source
df_joined["payer_source"] = df_joined["_merge"].map({
    "left_only": "Medicare Only",
    "right_only": "Medicaid Only",
    "both": "Both Payers"
})
df_joined.drop(columns=["_merge"], inplace=True)

# Fill NaN for numeric cols
numeric_cols = [c for c in df_joined.columns if c.startswith("medicare_") or c.startswith("medicaid_")]
df_joined[numeric_cols] = df_joined[numeric_cols].fillna(0)

# Reorder columns
col_order = [
    "npi", "provider_name", "provider_type", "provider_state",
    "hcpcs_code", "hcpcs_desc", "payer_source",
    "medicare_total_benes", "medicare_total_srvcs", "medicare_total_pymt",
    "medicare_total_chrg", "medicare_total_alowd",
    "medicaid_total_benes", "medicaid_total_claims", "medicaid_total_paid"
]
df_joined = df_joined[col_order]

# Summary stats
print(f"Joined dataset: {len(df_joined):,} rows")
print(f"Unique NPIs: {df_joined['npi'].nunique():,}")
print(f"Unique HCPCS codes: {df_joined['hcpcs_code'].nunique():,}")
print(f"\nPayer source breakdown:")
print(df_joined["payer_source"].value_counts().to_string())
print(f"\nSample:")
df_joined.head(10)

In [ ]:
# Chart 6: Payer source breakdown of NPI-HCPCS pairs
source_counts = df_joined["payer_source"].value_counts().reset_index()
source_counts.columns = ["Payer Source", "NPI-HCPCS Pairs"]

fig = px.pie(source_counts, values="NPI-HCPCS Pairs", names="Payer Source",
             title="NPI-HCPCS Pair Coverage: Medicare vs Medicaid Overlap",
             color="Payer Source",
             color_discrete_map={"Medicare Only": "#636EFA", "Medicaid Only": "#EF553B", "Both Payers": "#00CC96"})
fig.show()

---
## 4. Proof: Query NPI to Get All HCPCS Codes across Both Payers

In [ ]:
# Find a provider that appears in BOTH payers for a compelling demo
both_npis = df_joined[df_joined["payer_source"] == "Both Payers"]["npi"].value_counts()
# Pick an NPI with many HCPCS codes across both payers
demo_npi = both_npis.index[0] if len(both_npis) > 0 else df_joined["npi"].value_counts().index[0]
demo_name = medicare_provider_info.get(demo_npi, {}).get("provider_name", "Unknown")
demo_type = medicare_provider_info.get(demo_npi, {}).get("provider_type", "Unknown")
demo_state = medicare_provider_info.get(demo_npi, {}).get("provider_state", "")

print(f"Demo Provider: {demo_name}")
print(f"NPI: {demo_npi}")
print(f"Type: {demo_type} | State: {demo_state}")
print("=" * 80)

provider_df = df_joined[df_joined["npi"] == demo_npi].sort_values("hcpcs_code").reset_index(drop=True)

print(f"\nTotal unique HCPCS codes: {len(provider_df)}")
print(f"  - Medicare only: {(provider_df['payer_source'] == 'Medicare Only').sum()}")
print(f"  - Medicaid only: {(provider_df['payer_source'] == 'Medicaid Only').sum()}")
print(f"  - Both payers:   {(provider_df['payer_source'] == 'Both Payers').sum()}")
print(f"\nMedicare total payment: ${provider_df['medicare_total_pymt'].sum():,.2f}")
print(f"Medicaid total paid:    ${provider_df['medicaid_total_paid'].sum():,.2f}")
print(f"\nAll HCPCS codes for this provider:")
provider_df[["hcpcs_code", "hcpcs_desc", "payer_source",
             "medicare_total_srvcs", "medicare_total_pymt",
             "medicaid_total_claims", "medicaid_total_paid"]]

In [ ]:
# Reusable lookup function — query any NPI
def lookup_npi(npi_str):
    """Query all HCPCS codes for a given NPI across both payers."""
    npi_str = str(npi_str).strip()
    result = df_joined[df_joined["npi"] == npi_str].sort_values("hcpcs_code").reset_index(drop=True)
    if len(result) == 0:
        print(f"No records found for NPI: {npi_str}")
        return None
    info = medicare_provider_info.get(npi_str, {})
    print(f"Provider: {info.get('provider_name', 'N/A')} | Type: {info.get('provider_type', 'N/A')} | State: {info.get('provider_state', 'N/A')}")
    print(f"NPI: {npi_str} | Total HCPCS codes: {len(result)}")
    print(f"  Medicare only: {(result['payer_source'] == 'Medicare Only').sum()} | Medicaid only: {(result['payer_source'] == 'Medicaid Only').sum()} | Both: {(result['payer_source'] == 'Both Payers').sum()}")
    return result[["hcpcs_code", "hcpcs_desc", "payer_source",
                    "medicare_total_srvcs", "medicare_total_pymt",
                    "medicaid_total_claims", "medicaid_total_paid"]]

# Example: query the first NPI from the joined dataset
sample_npi = df_joined["npi"].iloc[0]
lookup_npi(sample_npi)